# Week 3 - Module 3: ECG Waveform Analysis
### 1D-CNN (Kernels, Padding, Stride) + Lightweight Self-Attention

Real data source: **MIT-BIH Arrhythmia Database** (PhysioNet, `mitdb`) - open access, no credentialing needed. Falls back automatically to a synthetic-but-ECG-shaped dataset if the download fails in this session, so the notebook never gets blocked by a network issue.

Run all cells top to bottom.

In [ ]:
import os
import sys
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/Cardiac_Risk_Prediction'
except ImportError:
    PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())

os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/outputs', exist_ok=True)
if f'{PROJECT_DIR}/src' not in sys.path:
    sys.path.append(f'{PROJECT_DIR}/src')

print(f'Working directory permanently mapped to: {PROJECT_DIR}')


In [ ]:
# Install wfdb (PhysioNet's WaveForm DataBase toolkit) - not preinstalled in Colab
!pip install -q wfdb


In [ ]:
import sys
sys.path.append(f'{PROJECT_DIR}/src')

from ecg_data import load_ecg_beats
from ecg_models import ECG1DCNN


## Load MIT-BIH ECG beats
Downloads a few MIT-BIH records (100, 101, 103, 106 - a standard mixed normal/arrhythmia subset) on first run, segments each beat around its annotated R-peak, and labels it Normal (0) or Abnormal/arrhythmic (1). If this fails for any reason, it prints why and switches to the synthetic fallback automatically - check the printed message to see which one you actually got.

In [ ]:
X, y = load_ecg_beats(data_dir=f'{PROJECT_DIR}/data/mitbih')
print(f"Beats: {len(X)} | Normal: {int((y==0).sum())} | Abnormal: {int((y==1).sum())}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visual sanity check: plot one normal and one abnormal beat
normal_idx = np.where(y == 0)[0][0]
abnormal_idx = np.where(y == 1)[0][0]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(X[normal_idx]); axes[0].set_title('Example Normal Beat')
axes[1].plot(X[abnormal_idx]); axes[1].set_title('Example Abnormal Beat')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/outputs/module3_example_beats.png', dpi=150, bbox_inches='tight')
plt.show()


## Split, standardize, and build DataLoaders

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Per-beat standardization (zero mean, unit variance across each window)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)

train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train).unsqueeze(1))
val_dataset = TensorDataset(torch.tensor(X_val), torch.tensor(y_val).unsqueeze(1))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Data ready! Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


## Build, train (or load checkpoint), and validate ECG1DCNN

In [ ]:
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import roc_auc_score, accuracy_score

CHECKPOINT_PATH = f'{PROJECT_DIR}/checkpoints/module3_ecg_cnn.pt'

model = ECG1DCNN(input_length=X_train.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

if os.path.exists(CHECKPOINT_PATH):
    print("Found pre-trained checkpoint in Drive! Loading weights (Skipping retraining)...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded checkpoint with Best Val AUC: {checkpoint['val_auc']:.4f}")
else:
    print("No checkpoint found. Starting full PyTorch training loop...")
    epochs = 40
    best_val_auc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'val_auc': []}

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            preds = model(batch_x)
            loss = criterion(preds, batch_y)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        total_val_loss = 0.0
        val_preds, val_targets = [], []
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                preds = model(batch_x)
                loss = criterion(preds, batch_y)
                total_val_loss += loss.item()
                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(batch_y.cpu().numpy())
        avg_val_loss = total_val_loss / len(val_loader)
        current_val_auc = roc_auc_score(val_targets, val_preds)

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_auc'].append(current_val_auc)

        if current_val_auc > best_val_auc:
            best_val_auc = current_val_auc
            torch.save({'model_state_dict': model.state_dict(),
                        'val_auc': best_val_auc}, CHECKPOINT_PATH)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1:02d}/{epochs}] | Train Loss: {avg_train_loss:.4f} | "
                  f"Val Loss: {avg_val_loss:.4f} | Val AUC: {current_val_auc:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['train_loss'], label='Train Loss')
    axes[0].plot(history['val_loss'], label='Val Loss')
    axes[0].set_title('Loss Curve'); axes[0].set_xlabel('Epochs'); axes[0].legend()
    axes[1].plot(history['val_auc'], label='Val AUC', color='green')
    axes[1].set_title('Validation AUC-ROC'); axes[1].set_xlabel('Epochs'); axes[1].legend()
    plt.savefig(f'{PROJECT_DIR}/outputs/week3_loss_auc.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Checkpoint and training plots successfully saved to Google Drive.")


## Final validation metrics

In [ ]:
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for batch_x, batch_y in val_loader:
        batch_x = batch_x.to(device)
        preds = model(batch_x)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(batch_y.numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
binary_preds = (all_preds > 0.5).astype(int)

print("=" * 45)
print(f"Final Validation Accuracy : {accuracy_score(all_targets, binary_preds) * 100:.2f}%")
print(f"Final Validation AUC-ROC  : {roc_auc_score(all_targets, all_preds):.4f}")
print(f"Model saved at            : {CHECKPOINT_PATH}")
print("=" * 45)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("\n" + "="*50)
print("MODULE 3 - ECG CLASSIFICATION REPORT")
print("="*50)
print(classification_report(all_targets, binary_preds, target_names=['Normal Beat', 'Abnormal Beat']))

cm = confusion_matrix(all_targets, binary_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Abnormal'])
disp.plot(cmap=plt.cm.Blues)
plt.title("Module 3: ECG Confusion Matrix")
plt.savefig(f'{PROJECT_DIR}/outputs/module3_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


## Notes for the progress report
- Confirm from the printed message above whether this run used **real MIT-BIH data** or the **synthetic fallback** - report that honestly, it changes how the results should be interpreted.
- The `pooled` 64-dim feature vector returned by `model(x, return_features=True)` is what Week 6's fusion layer will consume from this branch.